# 03 - Model training

Trains and evaluates the classifiers on the preprocessed training tables, including cross-validation, the dose-label comparison, ablations, noise tests and SHAP. Cell order is preserved from the original `habitability_pipeline.ipynb`; outputs are cleared.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

FEATS = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "star_type"]
LAB   = {"Low": 0, "Medium": 1, "High": 2}
NAMES = ["Low", "Medium", "High"]

X = train[FEATS].astype(float).values
y = train["label"].map(LAB).values
base = train["baseline_label"].map(LAB).values
groups = train["star_id"].values
obs_mask = (train["rate_basis"] == "count").values

def fit_predict(name, Xtr, ytr, Xte):
    imp = SimpleImputer(strategy="median")            # max_logE is NaN for Howard rows
    Xtr = imp.fit_transform(Xtr); Xte = imp.transform(Xte)
    if name == "RandomForest":
        m = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
        return m.predict(Xte)
    if name == "XGBoost":
        cls = np.unique(ytr); mp = {c: i for i, c in enumerate(cls)}
        m = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, eval_metric="mlogloss", random_state=SEED)
        m.fit(Xtr, np.array([mp[c] for c in ytr]), sample_weight=compute_sample_weight("balanced", ytr))
        return cls[m.predict(Xte)]
    if name == "MLP":
        sc = StandardScaler().fit(Xtr)
        m = MLPClassifier((128, 64, 32), max_iter=500, random_state=SEED).fit(sc.transform(Xtr), ytr)
        return m.predict(sc.transform(Xte))

splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups))

def cv_oof(name, cols=None):
    cols = list(range(X.shape[1])) if cols is None else cols
    oof = np.zeros(len(y), int)
    for tr, te in splits:
        oof[te] = fit_predict(name, X[tr][:, cols], y[tr], X[te][:, cols])
    return oof

def mf1(pred, mask): return f1_score(y[mask], pred[mask], average="macro", labels=[0, 1, 2], zero_division=0)

oof = {k: cv_oof(k) for k in ["RandomForest", "XGBoost", "MLP"]}
allm = np.ones(len(y), bool)
res = pd.DataFrame({"macroF1_all_rows":      {**{"Tilley-style rule": mf1(base, allm)}, **{k: mf1(v, allm) for k, v in oof.items()}},
                    "macroF1_observed_only": {**{"Tilley-style rule": mf1(base, obs_mask)}, **{k: mf1(v, obs_mask) for k, v in oof.items()}}}).round(3)
print(res)
print("rows: all =", len(y), "| observed (count-based) =", obs_mask.sum())

In [ ]:
best = res.loc[["RandomForest", "XGBoost", "MLP"], "macroF1_observed_only"].idxmax()
print("best model on observed rows:", best)

for title, mask in (("OBSERVED rows (count-based)", obs_mask), ("ALL rows", allm)):
    print("\n=====", title, "=====")
    print(classification_report(y[mask], oof[best][mask], labels=[0, 1, 2], target_names=NAMES, zero_division=0))
    print(pd.DataFrame(confusion_matrix(y[mask], oof[best][mask], labels=[0, 1, 2]),
                       index=[f"true {n}" for n in NAMES], columns=[f"pred {n}" for n in NAMES]))
print("\nMedium rows (observed):", int(((y == 1) & obs_mask).sum()), "-> report Medium as unreliable if this is small")

In [ ]:
full = mf1(cv_oof("RandomForest"), obs_mask)
rows = []
for i, f in enumerate(FEATS):
    keep = [j for j in range(len(FEATS)) if j != i]
    rows.append({"dropped": f, "macroF1_observed": mf1(cv_oof("RandomForest", keep), obs_mask)})
abl = pd.DataFrame(rows); abl["change_vs_full"] = abl["macroF1_observed"] - full
print("full model:", round(full, 3)); print(abl.round(3))

In [ ]:
import shap, matplotlib.pyplot as plt

imp = SimpleImputer(strategy="median"); Xi = imp.fit_transform(X)
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1).fit(Xi, y)
sv = shap.TreeExplainer(rf).shap_values(Xi)

def class_shap(sv, k):                       # handles both old (list) and new (3-D array) SHAP outputs
    return sv[k] if isinstance(sv, list) else sv[:, :, k]

imp_tab = pd.DataFrame({n: np.abs(class_shap(sv, k)).mean(0) for k, n in enumerate(NAMES)}, index=FEATS)
print(imp_tab.round(4))                       # mean |SHAP| per class: distance and flare rate should dominate High

shap.summary_plot(class_shap(sv, 2), Xi, feature_names=FEATS, show=False)
plt.title("SHAP summary: High class (RandomForest)"); plt.tight_layout(); plt.show()

In [ ]:
import os
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

FOLDER = r"D:\Research\data"
SEED = 42
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)      # show every column
df = pd.read_csv(os.path.join(FOLDER, "training_v11_split.csv"))
print(df.shape); print(df.columns.tolist())

# ---------- the fixed train/test split saved in the file (never re-split here) ----------
split_col = next((c for c in ["split", "set", "subset", "is_test", "fold", "test"] if c in df.columns), None)
if split_col is None:
    raise SystemExit("No split column found. Paste the column list above and I will adapt this cell.")
s = df[split_col]
if split_col == "fold":                                   is_test = s == 0
elif pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s): is_test = s.astype(bool)
else:                                                     is_test = s.astype(str).str.lower().isin(["test", "val", "validation", "holdout"])
print(f"split column '{split_col}':", s.value_counts().to_dict(), "| test rows:", int(is_test.sum()))
assert not set(df.loc[is_test, "star_id"]) & set(df.loc[~is_test, "star_id"]), "a star is in both train and test"

# ---------- features and labels ----------
if "max_logE_missing" not in df.columns: df["max_logE_missing"] = df["max_logE"].isna().astype(int)
FEATS = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "max_logE_missing", "star_type"]
LAB = {"Low": 0, "Medium": 1, "High": 2}; NAMES = ["Low", "Medium", "High"]
df["max_logE"] = df["max_logE"].fillna(df.loc[~is_test, "max_logE"].median())        # train median, no test information
X = df[FEATS].astype(float).values
y = df["label"].map(LAB).values
tr, te = (~is_test).values, is_test.values

def fit_predict(name, Xtr, ytr, Xte):
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=SEED, n_jobs=-1).fit(Xtr, ytr).predict(Xte)
    if name == "XGBoost":
        cls = np.unique(ytr); mp = {c: i for i, c in enumerate(cls)}
        m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, eval_metric="mlogloss", random_state=SEED)
        m.fit(Xtr, np.array([mp[c] for c in ytr]), sample_weight=compute_sample_weight("balanced", ytr))
        return cls[m.predict(Xte)]
    sc = StandardScaler().fit(Xtr)
    return MLPClassifier((64, 32), max_iter=500, random_state=SEED).fit(sc.transform(Xtr), ytr).predict(sc.transform(Xte))

pred = {m: np.full(len(df), -1) for m in ["RandomForest", "XGBoost", "MLP"]}
for m in pred: pred[m][te] = fit_predict(m, X[tr], y[tr], X[te])
pred["Distance-blind rule"] = df["baseline_label"].map(LAB).values

# ---------- test-set scores on different subsets ----------
src = df["source"].astype(str).str.lower()
GROUPS = {"Pietras": "pietras", "Su2022": "su2022", "Pietras+Su2022": "pietras|su2022"}      # row groups to test with and without
subsets = {"all test": te}
for g, k in GROUPS.items():
    has = src.str.contains(k).values
    subsets[f"no {g}"] = te & ~has
    if g != "Pietras+Su2022": subsets[f"{g} only"] = te & has
if "obs_assumed" in df.columns: subsets["real obs time only"] = te & ~df["obs_assumed"].astype(bool).values
def mf1(p, mask): return f1_score(y[mask], p[mask], average="macro", labels=np.unique(y[mask]), zero_division=0)   # only classes present in the subset
print("\nMACRO F1 ON THE FIXED TEST SET (models within about 0.03 are tied)")
print(pd.DataFrame({k: {m: round(mf1(p, mk), 3) for m, p in pred.items()} for k, mk in subsets.items()}))
print("rows per subset:", {k: int(mk.sum()) for k, mk in subsets.items()})

best = max(["RandomForest", "XGBoost", "MLP"], key=lambda m: mf1(pred[m], te))
print("\nbest model on all test rows:", best)
print(classification_report(y[te], pred[best][te], labels=[0, 1, 2], target_names=NAMES, zero_division=0))
print(pd.DataFrame(confusion_matrix(y[te], pred[best][te], labels=[0, 1, 2]), index=[f"true {n}" for n in NAMES], columns=[f"pred {n}" for n in NAMES]))

rows = []
for sname in sorted(src[te].unique()):
    mk = te & (src == sname).values
    rows.append({"source": sname, "test rows": int(mk.sum()), "accuracy": round(accuracy_score(y[mk], pred[best][mk]), 3), "macro F1": round(mf1(pred[best], mk), 3)})
print("\nper source (best model):"); print(pd.DataFrame(rows).to_string(index=False))

# ---------- does training on these rows help or hurt? (same test rows: those outside the group) ----------
print()
for g, k in GROUPS.items():
    has = src.str.contains(k).values
    if (tr & has).any() and (te & ~has).any():
        p_no = fit_predict(best, X[tr & ~has], y[tr & ~has], X[te & ~has])
        p_all = fit_predict(best, X[tr], y[tr], X[te & ~has])
        f = lambda p: round(f1_score(y[te & ~has], p, average="macro", labels=np.unique(y[te & ~has]), zero_division=0), 3)
        print(f"{best} on test rows without {g}: trained without {g} {f(p_no)} | trained with {g} {f(p_all)}")

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v11_split.csv")
a["max_logE_missing"] = a.max_logE.isna().astype(int)
med = a.loc[(a.split == "train") & a.max_logE.notna(), "max_logE"].median()
a["max_logE"] = a.max_logE.fillna(med)
cls = ["Low", "Medium", "High"]; cmap = {c: i for i, c in enumerate(cls)}
a["y"] = a.label.map(cmap)
tr = a[a.split == "train"].reset_index(drop=True)
te = a[a.split == "test"].reset_index(drop=True)
F = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "max_logE_missing", "star_type"]

def make():
    return {"RF": RandomForestClassifier(500, class_weight="balanced", random_state=42),
            "XGB": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42),
            "MLP": make_pipeline(StandardScaler(), MLPClassifier((64, 32), max_iter=2000, random_state=42))}
mf = lambda m, X: f1_score(te.y, m.predict(X), average="macro")

# 1. ablation
sets = {"all features": F,
        "no koi_sma": [c for c in F if c != "koi_sma"],
        "no rate features": [c for c in F if c not in ("flare_rate_month", "flare_rate_all")],
        "no energy features": [c for c in F if c not in ("max_logE", "max_logE_missing")],
        "no star_type": [c for c in F if c != "star_type"],
        "formula inputs only": ["koi_sma", "flare_rate_month"]}
abl = {}
for name, cols in sets.items():
    row = {}
    for n, m in make().items():
        if n == "MLP": continue
        m.fit(tr[cols], tr.y); row[n] = round(mf(m, te[cols]), 3)
    abl[name] = row
print("ABLATION, macro F1 on the fixed test set")
print(pd.DataFrame(abl).T.to_string())

# 2. noisy-input test (models trained once on clean data)
fit = {n: m.fit(tr[F], tr.y) for n, m in make().items()}
rng = np.random.default_rng(42)
def perturb(df, rate_noise, s_dist):
    x = df[F].copy()
    if rate_noise:
        nb = df.n_big.to_numpy(float); na = df.n_all.to_numpy(float); om = df.obs_months.to_numpy(float)
        ok = ~np.isnan(nb)
        x.loc[ok, "flare_rate_month"] = rng.poisson(nb[ok]) / om[ok]
        oka = ~np.isnan(na)
        x.loc[oka, "flare_rate_all"] = rng.poisson(na[oka]) / om[oka]
    if s_dist:
        x["koi_sma"] = x.koi_sma * np.exp(rng.normal(0, s_dist, len(x)))
    return x
scen = {"clean": (False, 0), "rate noise (Poisson counts)": (True, 0),
        "distance noise 10%": (False, 0.10), "distance noise 30%": (False, 0.30),
        "rate + distance 30%": (True, 0.30)}
out = {}
for s, (rn, sd) in scen.items():
    reps = 1 if s == "clean" else 30
    out[s] = {n: f"{np.mean([mf(m, perturb(te, rn, sd)) for _ in range(reps)]):.3f}"
              for n, m in fit.items()}
print("\nNOISY INPUT, mean macro F1 (30 repeats)")
print(pd.DataFrame(out).T.to_string())

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v11_split.csv")
a["max_logE_missing"] = a.max_logE.isna().astype(int)
tr0 = a[a.split == "train"]
for c in ["max_logE", "Teff"]:
    a[c] = a[c].fillna(tr0[c].median())
cmap = {"Low": 0, "Medium": 1, "High": 2}
a["y"] = a.label.map(cmap)
tr, te = a[a.split == "train"], a[a.split == "test"]
G = ["Teff", "star_type", "max_logE", "max_logE_missing"]   # no koi_sma, no flare rates

for name, m in {"majority class (dummy)": DummyClassifier(strategy="most_frequent"),
                "RF": RandomForestClassifier(500, class_weight="balanced", random_state=42),
                "XGB": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)}.items():
    m.fit(tr[G], tr.y)
    print(name, round(f1_score(te.y, m.predict(te[G]), average="macro"), 3))
print(classification_report(te.y, m.predict(te[G]), target_names=["Low", "Medium", "High"], digits=3))

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v11_split.csv")
tr0 = a[a.split == "train"]
a["Teff"] = a.Teff.fillna(tr0.Teff.median())
a["y"] = a.label.map({"Low": 0, "Medium": 1, "High": 2})
tr, te = a[a.split == "train"], a[a.split == "test"]
G = ["Teff", "star_type"]

for name, m in {"majority class (dummy)": DummyClassifier(strategy="most_frequent"),
                "RF": RandomForestClassifier(500, class_weight="balanced", random_state=42),
                "XGB": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)}.items():
    m.fit(tr[G], tr.y)
    print(name, round(f1_score(te.y, m.predict(te[G]), average="macro"), 3))
    if name == "RF":
        print(classification_report(te.y, m.predict(te[G]), target_names=["Low", "Medium", "High"], digits=3))

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

D = r"D:\Research\data"
a = pd.read_csv(rf"{D}\training_v11_split.csv")
a["max_logE_missing"] = a.max_logE.isna().astype(int)
med = a.loc[(a.split == "train") & a.max_logE.notna(), "max_logE"].median()
a["max_logE"] = a.max_logE.fillna(med)
cls = ["Low", "Medium", "High"]
a["y"] = a.label.map({c: i for i, c in enumerate(cls)})
tr = a[a.split == "train"].reset_index(drop=True)
te = a[a.split == "test"].reset_index(drop=True)
F = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "max_logE_missing", "star_type"]
m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42).fit(tr[F], tr.y)

# A. SHAP importance (XGBoost's built-in TreeSHAP)
c = m.get_booster().predict(xgb.DMatrix(te[F]), pred_contribs=True)[:, :, :-1]
imp = pd.DataFrame(np.abs(c).mean(axis=0).T, index=F, columns=cls)
imp["all classes"] = imp.mean(axis=1)
print("SHAP, mean |value| per feature (test rows)")
print(imp.round(3).sort_values("all classes", ascending=False).to_string())

# B. the formula applied to the same noisy inputs
rng = np.random.default_rng(42)
def perturb(df, rate_noise, s_dist):
    x = df[F].copy()
    if rate_noise:
        nb = df.n_big.to_numpy(float); na = df.n_all.to_numpy(float); om = df.obs_months.to_numpy(float)
        ok = ~np.isnan(nb); x.loc[ok, "flare_rate_month"] = rng.poisson(nb[ok]) / om[ok]
        oka = ~np.isnan(na); x.loc[oka, "flare_rate_all"] = rng.poisson(na[oka]) / om[oka]
    if s_dist:
        x["koi_sma"] = x.koi_sma * np.exp(rng.normal(0, s_dist, len(x)))
    return x
def formula_label(x):
    r = x["flare_rate_month"] * 0.47 * (0.16 / x["koi_sma"]) ** 2
    return np.where(r >= 1, 2, np.where(r >= 0.1, 1, 0))
scen = {"clean": (False, 0), "rate noise (Poisson counts)": (True, 0), "distance noise 10%": (False, 0.10),
        "distance noise 30%": (False, 0.30), "rate + distance 30%": (True, 0.30)}
rows = {}
for s, (rn, sd) in scen.items():
    reps = 1 if s == "clean" else 30
    rows[s] = {"formula": round(np.mean([f1_score(te.y, formula_label(perturb(te, rn, sd)), average="macro")
                                         for _ in range(reps)]), 3),
               "XGB": round(np.mean([f1_score(te.y, m.predict(perturb(te, rn, sd)), average="macro")
                                     for _ in range(reps)]), 3)}
print("\nNOISY INPUT, macro F1 (30 repeats)")
print(pd.DataFrame(rows).T.to_string())

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

D = r"D:\Research\data"
df = pd.read_csv(rf"{D}\training_v12_dose.csv")
M = {"Low": 0, "Medium": 1, "High": 2}
df["y"] = df["label_dose"].map(M)
df["yf"] = df["label"].map(M)            # formula label, comparison predictor only

tr = df[df.split == "train"].copy(); te = df[df.split == "test"].copy()
print("train", len(tr), "test", len(te), "| star overlap:", len(set(tr.star_id) & set(te.star_id)))
print("test classes:", te.label_dose.value_counts().to_dict())

BASE = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "star_type"]
SETS = {"base": BASE, "base + cme_prob": BASE + ["cme_prob"],
        "no koi_sma": [f for f in BASE if f != "koi_sma"],
        "no rate features": [f for f in BASE if f not in ("flare_rate_month", "flare_rate_all")]}
print("NaN per feature:", df[BASE + ["cme_prob"]].isna().sum().to_dict())
for s, f in SETS.items():
    assert not tr[f].isna().any().any() and not te[f].isna().any().any(), f"NaN in {s}"

def models():
    return {"XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                                     random_state=42, eval_metric="mlogloss"),
            "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
            "MLP": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32),
                                                                 max_iter=2000, random_state=42)),
            "Majority dummy": DummyClassifier(strategy="most_frequent")}

def mf1(y, p):
    return f1_score(y, p, average="macro", labels=sorted(set(y) | set(p)), zero_division=0)

rows = []
for sname, feats in SETS.items():
    for mname, m in models().items():
        m.fit(tr[feats], tr["y"])
        rows.append((sname, mname, round(mf1(te["y"], m.predict(te[feats])), 3)))
rows.append(("formula label as predictor", "-", round(mf1(te["y"], te["yf"]), 3)))
print(pd.DataFrame(rows, columns=["features", "model", "macro F1"]).to_string(index=False))

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

D = r"D:\Research\data"
M = {"Low": 0, "Medium": 1, "High": 2}
F = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE",
     "max_logE_missing", "star_type", "cme_prob"]

def mf1(y, p):
    return f1_score(y, p, average="macro", labels=sorted(set(y) | set(p)), zero_division=0)

def mk():
    return (XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                          random_state=42, eval_metric="mlogloss"),
            RandomForestClassifier(n_estimators=300, random_state=42))

def run(path, labcol, name):
    df = pd.read_csv(path)
    df["max_logE_missing"] = df["max_logE"].isna().astype(int)
    tr = df[df.split == "train"].copy(); te = df[df.split == "test"].copy()
    med = tr["max_logE"].median()
    tr["max_logE"] = tr["max_logE"].fillna(med); te["max_logE"] = te["max_logE"].fillna(med)
    ytr, yte = tr[labcol].map(M).values, te[labcol].map(M).values
    print(f"\n===== {name}: train {len(tr)}, test {len(te)}, star overlap "
          f"{len(set(tr.star_id) & set(te.star_id))} =====")

    # 1. Ablation (XGBoost / RandomForest macro F1)
    sets = {
        "all features": F,
        "no cme_prob": [f for f in F if f != "cme_prob"],
        "no koi_sma": [f for f in F if f != "koi_sma"],
        "no flare_rate_month": [f for f in F if f != "flare_rate_month"],
        "no rate features (both)": [f for f in F if f not in ("flare_rate_month", "flare_rate_all")],
        "no max_logE": [f for f in F if f not in ("max_logE", "max_logE_missing")],
        "no star_type": [f for f in F if f != "star_type"],
        "koi_sma + flare_rate_month only": ["koi_sma", "flare_rate_month"],
    }
    rows = []
    for k, feats in sets.items():
        x, r = mk()
        x.fit(tr[feats], ytr); r.fit(tr[feats], ytr)
        rows.append((k, round(mf1(yte, x.predict(te[feats])), 3),
                        round(mf1(yte, r.predict(te[feats])), 3)))
    print("\nAblation (macro F1)")
    print(pd.DataFrame(rows, columns=["features", "XGBoost", "RF"]).to_string(index=False))

    # 2. SHAP importances (XGBoost TreeSHAP, mean |value| on the test rows)
    x, _ = mk(); x.fit(tr[F], ytr)
    c = x.get_booster().predict(xgb.DMatrix(te[F]), pred_contribs=True)   # (n, classes, F+1)
    a = np.abs(c[:, :, :-1]).mean(axis=0)                                  # (classes, F)
    sh = pd.DataFrame(a.T, index=F, columns=["Low", "Medium", "High"])
    sh["overall"] = sh.mean(axis=1)
    print("\nSHAP mean |value|")
    print(sh.sort_values("overall", ascending=False).round(3).to_string())

    # 3. Noisy-input test: lognormal noise on the test inputs, 30 repeats,
    #    scored against the ORIGINAL labels. Noise levels are stress settings, not measured errors.
    x, r = mk(); x.fit(tr[F], ytr); r.fit(tr[F], ytr)
    rng = np.random.default_rng(42)
    levels = {"clean": (0, 0), "rate noise 0.3": (0.3, 0), "distance noise 0.1": (0, 0.1),
              "distance noise 0.3": (0, 0.3), "rate 0.3 + distance 0.3": (0.3, 0.3)}
    out = []
    for lv, (rs, ds) in levels.items():
        sx, sr, sf = [], [], []
        for _ in range(30):
            n = te.copy()
            if rs:
                fr = rng.lognormal(0, rs, len(n))
                n["flare_rate_month"] *= fr; n["flare_rate_all"] *= fr
            if ds:
                n["koi_sma"] *= rng.lognormal(0, ds, len(n))
            sx.append(mf1(yte, x.predict(n[F]))); sr.append(mf1(yte, r.predict(n[F])))
            if labcol == "label":                      # formula recomputed on the noisy inputs
                R = n.flare_rate_month * n.cme_prob * (0.16 / n.koi_sma) ** 2
                sf.append(mf1(yte, np.where(R >= 1, 2, np.where(R >= 0.1, 1, 0))))
        out.append((lv, round(np.mean(sx), 3), round(np.mean(sr), 3),
                    round(np.mean(sf), 3) if sf else None))
    print("\nNoisy inputs (mean macro F1, 30 repeats)")
    print(pd.DataFrame(out, columns=["noise", "XGBoost", "RF", "formula"]).to_string(index=False))

run(rf"{D}\training_v12_model.csv", "label", "FORMULA label")
run(rf"{D}\training_v12_dose.csv", "label_dose", "DOSE label")

In [ ]:
import pandas as pd, numpy as np, joblib
from xgboost import XGBClassifier

df = pd.read_csv(r"D:\Research\data\training_v12_dose.csv")
BASE = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "star_type"]
cls = {"Low": 0, "Medium": 1, "High": 2}

tr = df[df.split == "train"].copy()
med = tr["max_logE"].median()
tr["max_logE"] = tr["max_logE"].fillna(med)

m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                  random_state=42, eval_metric="mlogloss")
m.fit(tr[BASE], tr["label_dose"].map(cls))

joblib.dump({"model": m, "features": BASE, "classes": ["Low", "Medium", "High"],
             "max_logE_median": float(med)},
            r"D:\Research\data\xgb_dose_model.joblib")
print("saved", len(tr), "rows")

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

df = pd.read_csv(r"D:\Research\data\training_v12_dose.csv")
M = {"Low": 0, "Medium": 1, "High": 2}
df["y"] = df["label_dose"].map(M)
tr = df[df.split == "train"]; te = df[df.split == "test"]

BASE = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "star_type"]
SETS = {"all 5 inputs": BASE,
        "without flare_rate_all": [f for f in BASE if f != "flare_rate_all"],
        "without flare_rate_month": [f for f in BASE if f != "flare_rate_month"]}

for name, f in SETS.items():
    m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                      random_state=42, eval_metric="mlogloss")
    m.fit(tr[f], tr["y"])
    print(f"{name:28s} macro F1 = {f1_score(te['y'], m.predict(te[f]), average='macro'):.3f}")

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

D = r"D:\Research\data"
M = {"Low": 0, "Medium": 1, "High": 2}
BASE = ["koi_sma", "flare_rate_month", "flare_rate_all", "max_logE", "star_type"]

def models():
    return {"XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                                     random_state=42, eval_metric="mlogloss"),
            "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
            "MLP": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32),
                                                                 max_iter=2000, random_state=42)),
            "Always guess most common class": DummyClassifier(strategy="most_frequent")}

def scores(path, labcol):
    df = pd.read_csv(path)
    tr, te = df[df.split == "train"].copy(), df[df.split == "test"].copy()
    for d in (tr, te):                       # same fill as the formula-label run
        d["max_logE"] = d["max_logE"].fillna(tr["max_logE"].median())
    out = {}
    for n, m in models().items():
        m.fit(tr[BASE], tr[labcol].map(M))
        out[n] = f1_score(te[labcol].map(M), m.predict(te[BASE]), average="macro")
    return out, te

f_sc, _ = scores(rf"{D}\training_v12_model.csv", "label")
d_sc, te_d = scores(rf"{D}\training_v12_dose.csv", "label_dose")
formula_as_pred = f1_score(te_d["label_dose"].map(M), te_d["label"].map(M), average="macro")

tab = pd.DataFrame({"Formula label": f_sc, "Dose label": d_sc})
tab.loc["Formula label as the predictor"] = [np.nan, formula_as_pred]
tab.index.name = "Model"
tab = tab.round(3)

tab.to_csv(rf"{D}\results_table.csv")
tab.style.format("{:.3f}", na_rep="-").set_caption("Results (macro F1, fixed test set)")